#### QUERY AND VALIDATE `GIZMO.BRONZE.ORDERS`

In [0]:
orders_df = spark.table('''GIZMO.BRONZE.ORDERS_VW''')
display(orders_df)
display(f'Record Count: {orders_df.count()}')

#### PARSE ARRAY TYPE FOR `ORDERS`
- EXTRACT AND CAST KEY ORDER FIELDS FROM RAW JSON DATA

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

# 2. Apply the regex fix to the 'value' column first
# This addresses the order_date formatting issue
fixed_order_date_df = orders_df.withColumn(
    "fixed_value", 
    F.regexp_replace(F.col("value"), '"order_date": (\d{4}-\d{2}-\d{2})', '"order_date": "$1"')
)

# 3. Extract fields from the JSON string
# Note: Spark's get_json_object returns strings, so we cast to the desired types
fixed_order_df = fixed_order_date_df.select(
    F.get_json_object("value", "$.order_id").cast(IntegerType()).alias("order_id"),
    F.get_json_object("value", "$.customer_id").cast(IntegerType()).alias("customer_id"),
    F.col("fixed_value"),
    F.get_json_object("value", "$.transaction_timestamp").cast(TimestampType()).alias("transaction_timestamp"),
    F.get_json_object("value", "$.total_amount").cast(DoubleType()).alias("total_amount"),
    F.get_json_object("value", "$.payment_method").alias("payment_method"),
    F.get_json_object("value", "$.items[0].item_id").cast(IntegerType()).alias("item_1"),
    F.get_json_object("value", "$.items[1].item_id").cast(IntegerType()).alias("item_2")
)

display(fixed_order_df)

In [0]:
from pyspark.sql.types import DateType

fixed_order_df = fixed_order_date_df.select(
    F.get_json_object("value", "$.order_id").cast(IntegerType()).alias("order_id"),
    F.get_json_object("value", "$.customer_id").cast(IntegerType()).alias("customer_id"),
    
    # Extracting from the repaired JSON column
    F.get_json_object("fixed_value", "$.order_date").cast(DateType()).alias("order_date"),
    
    F.col("fixed_value"),
    F.get_json_object("value", "$.transaction_timestamp").cast(TimestampType()).alias("transaction_timestamp"),
    F.get_json_object("value", "$.total_amount").cast(DoubleType()).alias("total_amount"),
    F.get_json_object("value", "$.payment_method").alias("payment_method"),
    F.get_json_object("value", "$.items[0].item_id").cast(IntegerType()).alias("first_item_id")
)

display(fixed_order_df)

In [0]:
from pyspark.sql import functions as F

# 1. Define the schema
items_schema = "array<struct<item_id:int,quantity:int,price:double>>"

# 2. Transform
# Note: Included 'value' in the select so withColumn can use it
fixed_order_items_df = (
    fixed_order_df.select(
        'order_id', 
        'customer_id', 
        'order_date', 
        'fixed_value', 
        'transaction_timestamp',
        'total_amount', 
        'payment_method'
    )
    .withColumn(
        "items", 
        F.from_json(F.get_json_object(F.col("fixed_value"), "$.items"), items_schema)
    )
)

display(fixed_order_items_df)

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
# Create a temporary view named "v_fixed_orders"
fixed_order_items_df.createOrReplaceTempView("fixed_orders_temp_vw")

# You can now query it using Spark SQL
fixed_orders_temp_vw_df = spark.sql("SELECT * FROM fixed_orders_temp_vw")
display(fixed_orders_temp_vw_df)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW ORDERS_VW_TEMP_VW
AS
SELECT
  from_json(
    fixed_value, 
    'struct<
      items:array<struct<item_id:bigint,name:string,price:bigint,quantity:bigint>>,
      customer_id:int,
      order_date:string,
      order_id:bigint,
      order_status:string,
      payment_method:string,
      total_amount:bigint,
      transaction_timestamp:string
    >'
  ) AS json_value
FROM
  fixed_orders_temp_vw;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
%sql
CREATE OR REPLACE TABLE GIZMO.SILVER.ORDERS_JSON_DELTA
AS
SELECT * FROM ORDERS_VW_TEMP_VW;

In [0]:
%sql
SELECT
json_value.order_id::int AS order_id,
json_value.customer_id::int AS customer_id,
json_value.order_date::date AS order_date,
json_value.transaction_timestamp::timestamp AS transaction_timestamp,
json_value.total_amount::int AS total_amount,
json_value.payment_method::string AS payment_status,
explode(array_distinct(json_value.items)) AS item
FROM
GIZMO.SILVER.ORDERS_JSON_DELTA
ORDER BY 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ORDERS_ITEM_EXPLODE_TEMP_VW
AS
SELECT
json_value.customer_id::INT AS customer_id,
json_value.order_id::INT AS order_id,
json_value.order_date::DATE AS order_date,
json_value.order_status::STRING AS order_status,
json_value.payment_method::STRING AS payment_method,
json_value.total_amount::INT AS total_amount,
json_value.transaction_timestamp::TIMESTAMP AS transaction_timestamp,
explode(array_distinct(json_value.items)) AS item
FROM
GIZMO.SILVER.ORDERS_JSON_DELTA;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS_DELTA


In [0]:
%sql
CREATE OR REPLACE TABLE GIZMO.SILVER.ORDERS_DELTA
AS
SELECT
order_id,
customer_id,
item.item_id,
item.name,
order_date,
order_status,
item.price,
item.quantity,
payment_method,
total_amount,
transaction_timestamp
 FROM ORDERS_ITEM_EXPLODE_TEMP_VW;

#### VALIDATE AND QUERY `GIZMO.SILVER.ORDERS`

In [0]:
%python
orders_count_df = spark.sql('''SELECT * FROM GIZMO.SILVER.ORDERS_DELTA''');
print(f'Row Count: {orders_count_df.count()}')

## 📝 CAPTURE AUDIT & OBSERVABILITY MECHANISM

Track and log every data pipeline run for transparency, traceability, and operational monitoring.  
This section ensures all data loads are auditable and pipeline health is observable.

In [0]:
from pyspark.sql import Row
from datetime import datetime
import uuid
import sys
import traceback
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType, TimestampType

# =====================================================================
# 1. INITIALIZE PIPELINE METADATA
# =====================================================================
# Capture start time at the absolute beginning of the execution
load_start_time = datetime.now()
pipeline_name = 'PySpark-02.TransformOrders'

# Establish default tracking states
status = "SUCCESS"
message = "Loaded Orders data into Silver Table"
record_count = 0

try:
    # =====================================================================
    # 2. CORE ETL LOGIC
    # =====================================================================
    
    # Step A: Extract Data using absolute path and explicit format configuration
    orders_df = (spark.read.table('GIZMO.BRONZE.ORDERS_VW')) 
    
    # Step B: Execute target transformations or loading actions here
    # (Example: customers_df.write.mode("overwrite").saveAsTable("GIZMO.BRONZE.CUSTOMERS"))
    
    # Step C: Capture final evaluated source record count
    record_count = orders_df.count()
    
    # =====================================================================

except Exception as e:
    # 3. EXCEPTION HANDLING
    # If any error occurs above, catch it, flip status, and parse the trace
    status = "FAILED"
    
    exc_type, exc_value, exc_tb = sys.exc_info()
    error_details = traceback.format_exception_only(exc_type, exc_value)[0].strip()
    message = f"Pipeline failed! Error: {error_details}"
    record_count = -1  # Standard indicator flag representing an uncompleted execution

finally:
    # =====================================================================
    # 4. AUDIT & LOGGING (Guaranteed execution via finally block)
    # =====================================================================
    load_end_time = datetime.now()
    current_date = load_end_time.date()

    # Step A: Calculate Sequential Run ID for Today (Scoped to THIS specific pipeline)
    try:
        # Added F.lit() for reliable type matching and scoped the count to pipeline_name
        max_run_df = spark.table("GIZMO.AUDIT.AUDIT_LOGS") \
            .filter(
                (F.col("event_time") == F.lit(current_date)) & 
                (F.col("pipeline_name") == F.lit(pipeline_name))
            ) \
            .select(F.max(F.col("run_id").cast("int")).alias("max_id"))
        
        max_id_row = max_run_df.collect()[0]
        next_run_int = (max_id_row["max_id"] + 1) if max_id_row["max_id"] is not None else 1
    except Exception:
        # Defaults to 1 if table is empty, uninitialized, or completely drops out
        next_run_int = 1

    # Apply 2-digit zero padding format string (e.g., 1 -> "01", 11 -> "11")
    run_id_str = f"{next_run_int:02d}"

    # Step B: Secure Notebook Cluster Context Metadata safely
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
        user_name = context.tags().apply("user")
    except Exception:
        notebook_path = "Unknown/Local"
        user_name = "System"

    # Step C: Package the metadata tracking Row
    log_entry = Row(
        log_id=str(uuid.uuid4()),
        run_id=run_id_str,                
        event_time=current_date,          
        event_type="FULL LOAD",
        source_table='GIZMO.BRONZE.ORDERS_VW',
        target_table="GIZMO.SILVER.ORDERS_DELTA",
        record_count=record_count,
        status=status,                    
        message=message,                   
        user_name=user_name,
        notebook_path=notebook_path,
        pipeline_name=pipeline_name,
        load_start_time=load_start_time,  
        load_end_time=load_end_time       
    )

    # Step D: Declare structured explicit Schema types matching Target DDL exactly
    log_schema = StructType([
        StructField("log_id", StringType(), True),
        StructField("run_id", StringType(), True),  
        StructField("event_time", DateType(), True),
        StructField("event_type", StringType(), True),
        StructField("source_table", StringType(), True),
        StructField("target_table", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
        StructField("user_name", StringType(), True),
        StructField("notebook_path", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("load_start_time", TimestampType(), True),
        StructField("load_end_time", TimestampType(), True)
    ])

    # Step E: Instantiate log DataFrame and append transactional trace record
    log_entry_df = spark.createDataFrame([log_entry], schema=log_schema)
    log_entry_df.write.format("delta").mode("append").saveAsTable("GIZMO.AUDIT.AUDIT_LOGS")
    print(f"[AUDIT LOGGED] Status: {status} | Run ID: {run_id_str} | Count: {record_count}")

    # Step F: Force a hard stop exception for workflow orchestrators if pipeline failed
    if status == "FAILED":
        raise RuntimeError(message)

In [0]:
%python
dbutils.notebook.exit("ORDERS LOADED INTO GIZMO.SILVER.ORDERS_DELTA")

#### VALIDATE AUDIT TABLE RECORD COUNT `GIZMO.BRONZE.AUDIT_LOGS`

In [0]:
%sql
SELECT 
  run_id, 
  event_time, 
  pipeline_name, 
  record_count,
  date_format(FROM_UTC_TIMESTAMP(load_start_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_start_time_ist, 
  date_format(FROM_UTC_TIMESTAMP(load_end_time, 'Asia/Kolkata'), 'yyyy-MM-dd HH:mm:ss') AS load_end_time_ist
FROM GIZMO.AUDIT.AUDIT_LOGS
WHERE pipeline_name = 'PySpark-02.TransformOrders'
ORDER BY pipeline_name, event_time DESC, run_id ASC;